In [ ]:
"""
LIBRARIES
"""

import pandas as pd
import requests
import json
import time
import tempfile
import zipfile
import boto3

from pathlib import Path
from urllib.parse import urljoin
from bs4 import BeautifulSoup

#%pip install boto3

In [ ]:
"""
FUNCTION
"""

def get_crime_data(forces, start_date, end_date, bucket_name, aws_region, s3_prefix):
    """
    Download crime data (CSV files) for selected police forces, between selected dates.
    and dates.

    Parameters
    ----------
    forces : list
        Police.uk force IDs.

    start_date : str
        First month in YYYY-MM format.

    end_date : str
        Last month in YYYY-MM format.

    Returns
    -------
    dict
        One pandas DataFrame for each police force.
    """

    # create AWS S3 region connection
    s3 = boto3.client("s3", region_name=aws_region)

    # 1. Open the custom-download page
    form_url = "https://data.police.uk/data/"           # URL
    session = requests.Session()                        # create a web session
    
    form_response = session.get(form_url, timeout=30)   # Send a GET request to retrieve the HTML form. 
    form_response.raise_for_status()                    # timeout=30 stops the request if the server does not respond.

    # 2. Extract the security token from the form
    soup = BeautifulSoup(form_response.text, "html.parser")                  # parse html
    token_element = soup.select_one("input[name='csrfmiddlewaretoken']")     # locate CSRF token

    if token_element is None:
        raise RuntimeError("Could not find the Police.uk security token")    # error check incase token cannot be found
    csrf_token = token_element["value"]                                      # extract element's value attribute

    # 3. Submit the selected dates and forces
    form_data = {
        "csrfmiddlewaretoken": csrf_token,
        "date_from": start_date,
        "date_to": end_date,
        "forces": forces,
        "include_crime": "on"     # outcomes and stop-and-search are not requested.
    }

    request_response = session.post(       # submit the completed form using an HTTP POST request
        form_url,
        data=form_data,
        headers={"Referer": form_url},
        timeout=60                         # 60 second count
    )
    request_response.raise_for_status()    # raise an exception

    # 4. Find the URL used to check download progress
    soup = BeautifulSoup(request_response.text, "html.parser")    # parse HTML
    config_element = soup.select_one("#download-config")          # Find the HTML element containing the download configuration

    if config_element is None:
        raise RuntimeError(
            "Police.uk did not create a download request. "
            "Check the force IDs and date range.")            # error check

    download_config = json.loads(config_element.get_text())     # extract JSON
    progress_url = urljoin(form_url, download_config["url"])

    # 5. Wait for Police.uk to generate the ZIP
    print("Police.uk is generating the download...")       # comment for user
    zip_url = None                                         # start with no zip URL

    for attempt in range(150):
        progress_response = session.get(progress_url, timeout=30)
        progress_response.raise_for_status()                            # raise exception if status request failed
        progress = progress_response.json()                             # JSON -> dictionary
        status = progress.get("status")

        if status == "ready":
            zip_url = progress["url"]
            break

        if status == "error":
            raise RuntimeError("Police.uk could not generate the download")
        time.sleep(2)

    if zip_url is None:
        raise TimeoutError("The download was not ready after five minutes")

    print("Download ready. Downloading ZIP...")      # comment for user

    # 6. Start downloading the generated ZIP
    zip_response = session.get(
        zip_url,
        stream=True,
        timeout=120      # allow 120 seconds
    )
    zip_response.raise_for_status()

    # 7. Store and open the ZIP safely on Windows
    with tempfile.TemporaryDirectory() as temp_directory:
        zip_path = (
            Path(temp_directory)
            / "police_crime_data.zip"
        )

        with zip_path.open("wb") as zip_file:
            for chunk in zip_response.iter_content(
                chunk_size=1024 * 1024):
                if chunk:
                    zip_file.write(chunk)

        print("ZIP downloaded. Uploading to S3 bucket...")
        
        results = {}     # create empty results dictionary

        with zipfile.ZipFile(zip_path) as archive:
            filenames = archive.namelist()

            for force in forces:
                force_files = [
                    filename
                    for filename in filenames
                    if filename.lower().endswith(
                        f"-{force}-street.csv"
                    )
                ]

                if not force_files:
                    print(
                        f"Warning: no files found for {force}"
                    )

                    results[force] = pd.DataFrame()
                    continue

                monthly_data = []

                # Process one monthly file at a time
                for filename in force_files:
                    csv_name = Path(filename).name

                    # Filename begins with YYYY-MM
                    file_month = csv_name[:7]
                    year, month = file_month.split("-")

                    # Create a partitioned location in S3
                    s3_key = (
                        f"{s3_prefix}/"
                        f"force={force}/"
                        f"year={year}/"
                        f"month={month}/"
                        f"{csv_name}"
                    )

                    # Upload the original CSV directly from the ZIP
                    with archive.open(filename) as csv_file:
                        s3.upload_fileobj(
                            csv_file,
                            bucket_name,
                            s3_key,
                            ExtraArgs={
                                "ContentType": "text/csv"
                            }
                        )

                    print(f"Uploaded: s3://{bucket_name}/{s3_key}")

                    # Reopen the CSV and read the monthly batch
                    with archive.open(filename) as csv_file:
                        month_df = pd.read_csv(csv_file)

                    monthly_data.append(month_df)

                # Combine the monthly batches for this force
                results[force] = pd.concat(
                    monthly_data,
                    ignore_index=True
                )

                print(
                    f"{force}: "
                    f"{len(force_files)} files, "
                    f"{len(results[force]):,} rows"
                )

    print("--------")
    print("Ingestion and S3 upload complete.")

    return results

In [ ]:
"""
CALL
"""

forces = [
    "metropolitan",
    "west-midlands",
    "south-wales",
    "sussex"
]

start = "2024-01"
end = "2026-01"

bucket_name = "rockborne-ch19-g1-crime"
aws_region = "us-west-2"  # Change if you selected another region

crime_data = get_crime_data(
    forces=forces,
    start_date=start,
    end_date=end,
    bucket_name=bucket_name,
    aws_region=aws_region,
    s3_prefix="raw/uk-police"
)